# deps.dev

_Part of the **Decay Before Archival** project — see `README.md` for full project context, setup instructions, and links to the other notebooks (`deps_dev.ipynb`, `scorecard.ipynb`, `gh_archive.ipynb`, `pypi.ipynb`, `other_sources.ipynb`)._

In [ ]:
from google.cloud import bigquery
from decay_before_archival import get_client

client = get_client()

## Explore: deps.dev

deps.dev publishes package/dependency metadata as a public BigQuery dataset. We don't know the exact table names yet, so first list what's in the dataset, then peek columns on whichever table looks relevant before writing real queries.

**Planned analysis:** `PackageVersions`/`ProjectStatus` for release cadence and deprecation flags, `Dependents`/`DependencyGraphEdges` for how many projects rely on a package (blast radius), `Projects` for repo-level status deps.dev has already enriched.

In [ ]:
# List tables in the deps.dev public dataset
tables_query = """
SELECT table_name
FROM `bigquery-public-data.deps_dev_v1.INFORMATION_SCHEMA.TABLES`
ORDER BY table_name
"""

job_config = bigquery.QueryJobConfig(dry_run=True, use_query_cache=False)
dry_run = client.query(tables_query, job_config=job_config)
print(f"This query will process {dry_run.total_bytes_processed / 1e6:.2f} MB")

client.query(tables_query).to_dataframe()

In [ ]:
# Pick a table name from the list above, then peek its columns and grab a small sample
table_name = "NPMRequirements"  # replace with a name from the list above

columns_query = f"""
SELECT column_name, data_type
FROM `bigquery-public-data.deps_dev_v1.INFORMATION_SCHEMA.COLUMNS`
WHERE table_name = '{table_name}'
ORDER BY ordinal_position
"""

job_config = bigquery.QueryJobConfig(dry_run=True, use_query_cache=False)
dry_run = client.query(columns_query, job_config=job_config)
print(f"This query will process {dry_run.total_bytes_processed / 1e6:.2f} MB")

client.query(columns_query).to_dataframe()

In [ ]:
sample_query = f"""
SELECT *
FROM `bigquery-public-data.deps_dev_v1.{table_name}`
LIMIT 10
"""

job_config = bigquery.QueryJobConfig(dry_run=True, use_query_cache=False)
dry_run = client.query(sample_query, job_config=job_config)
print(f"This query will process {dry_run.total_bytes_processed / 1e6:.2f} MB")

client.query(sample_query).to_dataframe()

## Coverage + features (two-year window)

Join key is `repo_full_name` (lowercased) ↔ `Projects.Name` / `PackageVersionToProject.ProjectName` (`Type = 'GITHUB'`).

- **Two snapshots:** `SNAP_END` = last weekly snapshot before the 2025-01-01 cutoff, `SNAP_START` = one year earlier. `Projects` stats (stars/forks/open issues) are taken at both so we can compute change. Release history comes from `PackageVersions` at `SNAP_END`, which already contains every version ever published (so 2023 and 2024 counts both come from one scan).
- **Don't use later snapshots for features.** `Deprecated` / `ProjectStatus` are empty until late 2025 — they're candidate *labels* (see the "Latest status" section), not features.
- Run order: setup cell → coverage → features → export. The dependents cell is opt-in (~420 GB scan).

In [ ]:
import pandas as pd

# Repos from the GH Archive feature set (label = inactive_next_365d, cutoff 2025-01-01).
# We match on LOWER() because GitHub names are case-insensitive.
repo_names = (
    pd.read_csv("../data/gh_archive_features_2024_clean.csv", usecols=["repo_full_name"])
    ["repo_full_name"].str.lower().unique().tolist()
)

# deps.dev snapshots are weekly. Pick the last one before the cutoff, and the last one a year earlier,
# so we can compute year-over-year change without looking past 2025-01-01 (no label leakage).
snap = next(iter(client.query("""
SELECT
  (SELECT MAX(Time) FROM `bigquery-public-data.deps_dev_v1.Snapshots` WHERE Time < TIMESTAMP('2025-01-01')) AS snap_end,
  (SELECT MAX(Time) FROM `bigquery-public-data.deps_dev_v1.Snapshots` WHERE Time < TIMESTAMP('2024-01-01')) AS snap_start
""").result()))
SNAP_END, SNAP_START = snap.snap_end, snap.snap_start
print(f"{len(repo_names):,} repos | snapshots: {SNAP_START} -> {SNAP_END}")


def run(sql, dry=False):
    """Run a query with the shared @repos / @snap_start / @snap_end params; dry=True just prints the scan size."""
    cfg = bigquery.QueryJobConfig(
        dry_run=dry,
        use_query_cache=not dry,
        query_parameters=[
            bigquery.ArrayQueryParameter("repos", "STRING", repo_names),
            bigquery.ScalarQueryParameter("snap_start", "TIMESTAMP", SNAP_START),
            bigquery.ScalarQueryParameter("snap_end", "TIMESTAMP", SNAP_END),
        ],
    )
    job = client.query(sql, job_config=cfg)
    if dry:
        print(f"This query will process {job.total_bytes_processed / 1e9:.2f} GB")
        return None
    return job.to_dataframe()

In [ ]:
COVERAGE = """
WITH p AS (
  SELECT SnapshotAt, LOWER(Name) AS repo
  FROM `bigquery-public-data.deps_dev_v1.Projects`
  WHERE SnapshotAt IN (@snap_start, @snap_end) AND Type = 'GITHUB' AND LOWER(Name) IN UNNEST(@repos)
),
pk AS (
  SELECT DISTINCT LOWER(ProjectName) AS repo
  FROM `bigquery-public-data.deps_dev_v1.PackageVersionToProject`
  WHERE SnapshotAt = @snap_end AND ProjectType = 'GITHUB' AND RelationType = 'SOURCE_REPO_TYPE'
    AND LOWER(ProjectName) IN UNNEST(@repos)
)
SELECT
  ARRAY_LENGTH(@repos) AS repos_total,
  (SELECT COUNT(*) FROM p WHERE SnapshotAt = @snap_end)   AS in_projects_end,
  (SELECT COUNT(*) FROM p WHERE SnapshotAt = @snap_start) AS in_projects_start,
  (SELECT COUNT(*) FROM p a JOIN p b USING (repo) WHERE a.SnapshotAt = @snap_end AND b.SnapshotAt = @snap_start) AS in_projects_both,
  (SELECT COUNT(*) FROM pk) AS with_published_package
"""

run(COVERAGE, dry=True)
run(COVERAGE).T

In [ ]:
FEATURES = """
WITH
proj AS (
  SELECT LOWER(Name) AS repo,
    MAX(IF(SnapshotAt = @snap_end,   StarsCount, NULL))      AS stars_end,
    MAX(IF(SnapshotAt = @snap_start, StarsCount, NULL))      AS stars_start,
    MAX(IF(SnapshotAt = @snap_end,   ForksCount, NULL))      AS forks_end,
    MAX(IF(SnapshotAt = @snap_start, ForksCount, NULL))      AS forks_start,
    MAX(IF(SnapshotAt = @snap_end,   OpenIssuesCount, NULL)) AS open_issues_end,
    MAX(IF(SnapshotAt = @snap_start, OpenIssuesCount, NULL)) AS open_issues_start
  FROM `bigquery-public-data.deps_dev_v1.Projects`
  WHERE SnapshotAt IN (@snap_start, @snap_end) AND Type = 'GITHUB' AND LOWER(Name) IN UNNEST(@repos)
  GROUP BY repo
),
pkg AS (
  SELECT LOWER(ProjectName) AS repo, System, Name, Version
  FROM `bigquery-public-data.deps_dev_v1.PackageVersionToProject`
  WHERE SnapshotAt = @snap_end AND ProjectType = 'GITHUB' AND RelationType = 'SOURCE_REPO_TYPE'
    AND LOWER(ProjectName) IN UNNEST(@repos)
),
base AS (  -- every repo deps.dev knows about, via Projects OR a package link (some have packages but no Projects row)
  SELECT repo FROM proj
  UNION DISTINCT
  SELECT repo FROM pkg
),
rel AS (
  SELECT k.repo, k.System, k.Name, v.UpstreamPublishedAt AS published_at,
         ARRAY_LENGTH(v.Advisories) AS n_advisories
  FROM pkg k
  JOIN `bigquery-public-data.deps_dev_v1.PackageVersions` v
    ON v.System = k.System AND v.Name = k.Name AND v.Version = k.Version
  WHERE v.SnapshotAt = @snap_end
)
SELECT
  b.repo,
  ANY_VALUE(p.repo IS NOT NULL)                                                   AS in_projects,
  p.stars_end, p.stars_start, p.forks_end, p.forks_start, p.open_issues_end, p.open_issues_start,
  COUNT(DISTINCT CONCAT(r.System, '/', r.Name))                                   AS n_packages,
  COUNT(DISTINCT r.System)                                                        AS n_ecosystems,
  COUNT(r.published_at)                                                           AS n_versions_total,
  COUNTIF(r.published_at >= TIMESTAMP('2024-01-01') AND r.published_at < TIMESTAMP('2025-01-01')) AS versions_2024,
  COUNTIF(r.published_at >= TIMESTAMP('2023-01-01') AND r.published_at < TIMESTAMP('2024-01-01')) AS versions_2023,
  DATE_DIFF(DATE '2025-01-01', DATE(MAX(r.published_at)), DAY)                    AS days_since_last_version_at_cutoff,
  SUM(r.n_advisories)                                                             AS advisory_links
FROM base b
LEFT JOIN proj p ON p.repo = b.repo
LEFT JOIN rel r ON r.repo = b.repo
GROUP BY b.repo, p.stars_end, p.stars_start, p.forks_end, p.forks_start, p.open_issues_end, p.open_issues_start
"""

run(FEATURES, dry=True)
deps_features = run(FEATURES)
deps_features.head()

In [ ]:
# Export for the integration notebook. One row per GH Archive repo that deps.dev knows about (join on repo_full_name_lower).
# Repos missing from this file are NOT in deps.dev -> report them as missing in the integration step.
deps_features.rename(columns={"repo": "repo_full_name_lower"}).to_csv("../data/deps_dev_features_2024.csv", index=False)
print(f"wrote {len(deps_features):,} rows")

## Latest status (label candidate — NOT a feature)

`Deprecated` / `ProjectStatus` are empty in 2024 snapshots, so we read them from the **latest** snapshot and keep them in a separate file. Use them only as an alternative outcome label, never as inputs for the 2025 cutoff. (Test run: 0 of the matched repos are `archived`, ~65 have a deprecated package.)

In [ ]:
SNAP_LATEST = next(iter(client.query("SELECT MAX(Time) AS t FROM `bigquery-public-data.deps_dev_v1.Snapshots`").result())).t

STATUS = """
WITH pkg AS (
  SELECT LOWER(ProjectName) AS repo, System, Name, Version
  FROM `bigquery-public-data.deps_dev_v1.PackageVersionToProject`
  WHERE SnapshotAt = @snap_latest AND ProjectType = 'GITHUB' AND RelationType = 'SOURCE_REPO_TYPE'
    AND LOWER(ProjectName) IN UNNEST(@repos)
)
SELECT k.repo,
  COUNT(*)                                 AS n_versions_latest,
  LOGICAL_OR(v.ProjectStatus = 'archived') AS any_archived,
  LOGICAL_OR(v.Deprecated IS NOT NULL)     AS any_deprecated,
  LOGICAL_AND(v.Deprecated IS NOT NULL)    AS all_deprecated
FROM pkg k
JOIN `bigquery-public-data.deps_dev_v1.PackageVersions` v
  ON v.System = k.System AND v.Name = k.Name AND v.Version = k.Version
WHERE v.SnapshotAt = @snap_latest
GROUP BY k.repo
"""

status_cfg = lambda dry: bigquery.QueryJobConfig(dry_run=dry, query_parameters=[
    bigquery.ArrayQueryParameter("repos", "STRING", repo_names),
    bigquery.ScalarQueryParameter("snap_latest", "TIMESTAMP", SNAP_LATEST),
])
job = client.query(STATUS, job_config=status_cfg(True))
print(f"This query will process {job.total_bytes_processed / 1e9:.2f} GB")  # ~32 GB

status_df = client.query(STATUS, job_config=status_cfg(False)).to_dataframe()
status_df.rename(columns={"repo": "repo_full_name_lower"}).to_csv("../data/deps_dev_status_latest.csv", index=False)
print(f"wrote {len(status_df):,} rows")
status_df[["any_archived", "any_deprecated", "all_deprecated"]].sum()

In [ ]:
# OPTIONAL / EXPENSIVE: direct-dependent counts (blast radius), rolled up to repo level, at the end snapshot.
# Dependents is huge (~420 GB scanned, ~40% of the 1 TB monthly free tier), so this is off by default.
RUN_DEPENDENTS = False

DEPENDENTS = """
WITH pkg AS (
  SELECT DISTINCT LOWER(ProjectName) AS repo, System, Name
  FROM `bigquery-public-data.deps_dev_v1.PackageVersionToProject`
  WHERE SnapshotAt = @snap_end AND ProjectType = 'GITHUB' AND RelationType = 'SOURCE_REPO_TYPE'
    AND LOWER(ProjectName) IN UNNEST(@repos)
)
SELECT pkg.repo,
       COUNT(DISTINCT CONCAT(d.Dependent.System, '/', d.Dependent.Name)) AS direct_dependents
FROM `bigquery-public-data.deps_dev_v1.Dependents` d
JOIN pkg USING (System, Name)
WHERE d.SnapshotAt = @snap_end AND d.MinimumDepth = 1
GROUP BY pkg.repo
"""

run(DEPENDENTS, dry=True)
if RUN_DEPENDENTS:
    dependents_df = run(DEPENDENTS)
    dependents_df.rename(columns={"repo": "repo_full_name_lower"}).to_csv("../data/deps_dev_dependents_2024.csv", index=False)
    print(f"wrote {len(dependents_df):,} rows")

**Notes:**

_(add observations here)_

## Which deps.dev tables we use (and why not the rest)

The proposal needs these variables: release cadence / time since last release, community signals (stars, forks), **number of projects that depend on this project**, and advisories as a health signal. deps.dev is only one source for that list; commits, bus factor, contributor turnover and issue/PR responsiveness come from GH Archive / the GitHub API, and Criticality Score + per-check scores come from OpenSSF Scorecard.

| Table | Decision | Why |
|---|---|---|
| `Projects` | **Used** | stars / forks / open issues at two snapshots (community engagement, year-over-year change) |
| `PackageVersionToProject` | **Used** | the only repo -> package link (the join to GitHub repos) |
| `PackageVersions` | **Used** | release cadence, time since last release, advisory links |
| `Snapshots` | **Used** | pick snapshots before the 2025-01-01 cutoff (no label leakage) |
| `Dependents` | **In scope, deferred** | "number of dependent projects" is in the proposal, but it scans ~420 GB (~40% of the monthly free tier). Opt-in cell below; run once we know which repos we keep |
| `Advisories` | Skipped | advisory data is planned from OSV.dev; `advisory_links` from `PackageVersions` is enough for a first pass |
| `Dependencies`, `DependencyGraphEdges` | Skipped | a package's own dependency tree isn't a proposal variable, and they scan 224 GB / 494 GB per snapshot |
| `*Requirements` (npm, PyPI, Maven, ...) | Skipped | per-ecosystem declared requirements; not a decay signal and only covers one ecosystem each |
| `PackageVersionHashes` | Skipped | file hashes, unrelated to project health |
| `*Latest` tables | **Never for features** | they reflect today's state, which would leak the 2025 outcome. Only the latest-status cell uses a recent snapshot, and only as a label candidate |

**Coverage caveat:** deps.dev only knows repos linked to a published package, so only ~2% of the GH Archive repos match. That's the "coverage gaps between data sources" the proposal says to be explicit about.

**Notes:**

- **Coverage:** deps.dev knows 1,405 of the 62,627 GH Archive rows (2.2%): 951 have a published package and 454 are listed with no package. Matching is on the lowercased repo name; renamed or recreated repos can silently fail to match.
- **Package-only repos:** 165 repos have a package link but no `Projects` row at the 2024-12-30 snapshot. They're kept (`in_projects = False`) with release data but `stars_*`, `forks_*` and `open_issues_*` are missing.
- **Missing `_start` columns:** 854 matched repos have no `*_start` values (the 165 above plus 689 repos that weren't in deps.dev a year earlier). That's structural, not random — don't impute or treat as zero.
- **Missing release dates:** `days_since_last_version_at_cutoff` is missing for 655 matched repos, but only 454 have no package, so roughly 201 repos with packages have no publish timestamp. _Unverified guess:_ an ecosystem (e.g. Go) that doesn't record `UpstreamPublishedAt`. Until checked, `versions_2023` / `versions_2024` are probably undercounts for those repos.
- **Not features:** `Deprecated` / `ProjectStatus` are empty before late 2025. In the latest snapshot none of the matched repos is `archived` and about 65 have a deprecated package, so `archived` is not usable as a label here.
- **Skipped on purpose:** see the table-decision section above. `Dependents` is in scope for the proposal but deferred (~420 GB).